# 🎬 CKD Video Studio — topic → finished video (free Colab, CPU only)

Type a topic. Get a narrated, animated explainer with one recurring character, in **English or Urdu**, in **16:9 or 9:16**.

**What this notebook does**
1. Checks the runtime (Python, ffmpeg, fonts, disk, TTS).
2. Writes a script + scene plan you can read and edit.
3. Shows storyboard stills so you can check the visuals before committing to a render.
4. Renders scene-by-scene, caches every scene, then muxes the final MP4.
5. Gives you a download link for the video and a ZIP of everything.

**Honest notes before you start**
- Everything runs **on the CPU** of a free Colab runtime. No GPU, no paid API, and **no hosted AI service** is contacted. Script writing is done by a local knowledge pack (or an optional local GGUF model), and narration by a local TTS voice.
- If a neural voice cannot be downloaded, the notebook **says so** and uses the offline robotic `espeak` voice. It never silently swaps voices.
- A 60 s 16:9 video at 720p takes roughly **8–12 minutes** on a free CPU runtime; 9:16 is about three times faster because the frame is smaller.
- Colab can disconnect. Everything is checkpointed per scene, so you can reopen the project and continue without re-rendering finished scenes.

## 0 · Get the code into the runtime

Pick **one** of these:

* **You cloned the repo already** (recommended): run the cell below with `REPO_URL` set to your repo and it will clone it into `/content/CKD`.
* **You have a ZIP** uploaded to Colab: upload it, then run `!unzip -q ckd.zip -d /content`.
* **The files are already here**: just `%cd` into the folder.

> Private repo? Use a GitHub token in the URL **only** in this cell, never in a file you commit — see `docs/PRIVATE_REPO.md`.

In [ ]:
import os, sys, subprocess, pathlib

# ── edit if needed ────────────────────────────────────────────────────────────
REPO_URL = ""          # e.g. "https://github.com/you/CKD.git"  (leave "" if already present)
PROJECT_ROOT = "/content/CKD"
# ──────────────────────────────────────────────────────────────────────────────

if REPO_URL and not pathlib.Path(PROJECT_ROOT, "ckd").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_ROOT], check=True)

if pathlib.Path(PROJECT_ROOT, "ckd").exists():
    os.chdir(PROJECT_ROOT)
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    print("repo:", PROJECT_ROOT)
else:
    print("⚠️  repo not found. Set REPO_URL above, or upload the project ZIP and unzip it to /content/CKD.")
print("cwd  :", os.getcwd())

## 1 · Install the dependencies (about 1–2 minutes)

CPU-only wheels. `imageio-ffmpeg` brings its own static `ffmpeg`, so you do **not** need `apt-get`.

In [ ]:
%cd /content/CKD
!pip install -q -r requirements.txt
print("dependencies installed")

## 2 · Health check

If `problems` lists anything, fix it before rendering — a half-configured runtime is the most common cause of a failed render.

In [ ]:
import importlib, json
import ckd.notebook_api as nb
importlib.reload(nb)

report = nb.check_environment("/content")
for key, value in report.items():
    if key != "problems":
        print(f"{key:>14}: {value}")
print()
if report["problems"]:
    print("⚠️  problems:")
    for p in report["problems"]:
        print("   -", p)
else:
    print("✅ environment looks good")

## 3 · Choose the topic and the settings

Any topic works. The three with hand-written, fact-checked scenes are **solar energy**, **how wifi works** and **bridge engineering**; other topics get a clearly-labelled generic scaffold (and the report says so).

In [ ]:
import pandas as pd
display(pd.DataFrame(nb.topics()))

In [ ]:
TOPIC      = "solar energy"     # try: "how wifi works", "bridge engineering"
LANGUAGES  = ["en"]             # ["en"], ["ur"], or ["en", "ur"] for both
FORMATS    = ["landscape"]      # ["landscape"] = 16:9, ["vertical"] = 9:16, or both
DURATION   = 45                 # target seconds (the script reports if it cannot hit it)
QUALITY    = "standard"         # "draft" (480p, fast), "standard" (720p), "high" (1080p)
CAPTIONS   = True
CHARACTER  = True

project = nb.new_project(
    TOPIC,
    languages=LANGUAGES,
    formats=FORMATS,
    duration=DURATION,
    quality=QUALITY,
    captions=CAPTIONS,
    character=CHARACTER,
    projects_dir="/content/projects",
    root="/content",
)
print("project folder:", project.root)

## 3b · Alternative: render a script you already wrote

Skip the topic composer entirely. Paste (or upload) a finished documentary script in the
usual format — `SCENE n — TITLE`, `**[VOICEOVER]**`, `**[VISUAL DIRECTION]**` with
timecodes, `**[ON-SCREEN TEXT]**`, optional `CALL TO ACTION` — and the engine speaks your
narration **verbatim** and draws a faceless business documentary from it.

Two honest limits, both reported in `warnings`: it draws **no footage** (visual direction
becomes labelled motion graphics), and it does **not pad** to the declared duration.
Format reference: `docs/YOUR_SCRIPT.md`; example: `scripts/nokia.md`.

In [ ]:
# Paste your script between the triple quotes (or upload it and pass the path).
# This is a three-scene mini documentary so the cell runs end to end; the full
# example is scripts/nokia.md in the repo.
SCRIPT = """
THE RISE AND FALL OF A PHONE GIANT

**Target duration:** 30 seconds

SCENE 1 — THE HOOK

**Time:** 0:00-0:20

**[VOICEOVER]**

Imagine building a company so powerful that millions of people carry your product.

A company whose name becomes almost synonymous with mobile phones.

This is how a giant lost its empire.

**[VISUAL DIRECTION]**

- 0:00-0:05: A classic mobile phone ringing in a dark room.
- 0:05-0:12: A world map illuminates as phone icons spread across continents.

**[ON-SCREEN TEXT]**

2007 - market leader

SCENE 2 — THE WORLD CHANGES

**Time:** 0:20-0:50

**[VOICEOVER]**

Then came a turning point. A different vision of what a phone could become.

A large touchscreen. A software-driven experience.

**[VISUAL DIRECTION]**

- 0:00-0:12: Cut to a touchscreen smartphone on a dark stage.
- 0:12-0:30: Display a software ecosystem: users, developers, apps, devices.

**[ON-SCREEN TEXT]**

From devices...

To ecosystems

SCENE 3 — THE LESSON

**Time:** 0:50-1:10

**[VOICEOVER]**

A strong product is not enough. The ecosystem matters.

And the companies that lasted were the ones that adapted.

**[VISUAL DIRECTION]**

- Show a split-screen comparison between a keypad phone and a touchscreen phone.

CALL TO ACTION

**[VOICEOVER]**

If you enjoy documentaries about the decisions that build empires, subscribe for more.

**[VISUAL DIRECTION]**

Display a clean subscribe animation over a subtle technology-themed background.
"""

from ckd.notebook_api import new_project_from_script

project_script = new_project_from_script(
    SCRIPT,                            # or: "scripts/nokia.md" (a path you uploaded)
    formats=("landscape", "vertical"),
    quality="draft",                   # "standard" for the watchable render
    voice_en="kokoro:am_michael",
)

for w in project_script.story().warnings:
    print("WARNING:", w)
# the rest of the notebook (review, storyboard, render, download) uses `project`
project = project_script
print("\nscenes:", len(project.story().scenes),
      "| project:", project.pipeline.project.root)


## 4 · Read the script (this is the part worth reviewing)

Every scene shows its narration in each language, its headline, and the visual elements it will use. Edit the JSON in the next cell if you want different words — the plan is re-validated before anything is rendered.

In [ ]:
for i, scene in enumerate(project.story().scenes, start=1):
    print(f"\n── scene {i}: {scene.id} ({scene.purpose}) ──")
    print("headline:", scene.headline.get(LANGUAGES[0], ""))
    for lang in LANGUAGES:
        print(f"  {lang}: {scene.narration[lang]}")
    print("  visuals:", ", ".join(e["type"] for e in scene.elements))
    if scene.keywords.get(LANGUAGES[0]):
        print("  keywords:", ", ".join(scene.keywords[LANGUAGES[0]]))

### 4b · (Optional) rewrite the script

Change `story_json` and run the cell. Leave it untouched to keep the generated plan.

In [ ]:
story_json = project.story_text()
print(story_json)

In [ ]:
# Re-validate and save whatever is in `story_json` above (skips if unchanged).
import json
try:
    edited = project.save_story(story_json)
    print("saved:", len(edited.scenes), "scenes")
except Exception as exc:
    print("❌ the edited script was rejected, nothing was written:", exc)

## 5 · Look at the storyboard before spending render time

One contact sheet per language × format. These are real frames from the renderer, not thumbnails of a template.

In [ ]:
from IPython.display import Image as IPImage, display

for path in nb.preview(project, samples=2):
    print(path)
    display(IPImage(filename=str(path), width=980 if "landscape" in path.name else 340))

## 6 · Render

Scenes are written as individual MP4s and cached. Re-running this cell after an edit only re-renders the scenes whose text or visuals changed.

In [ ]:
result = nb.render(project)
display(pd.DataFrame(nb.result_table(result)))
print("total wall time:", result["seconds"], "s")

## 7 · Watch the result

The player below is the actual rendered file from `/content`.

In [ ]:
from IPython.display import Video, display

for row in nb.result_table(result):
    print(row["video"], "·", row["seconds"], "s ·", row["size"], "·", row["fps"], "fps · verified:", row["verified"])
    display(Video(row["path"], embed=True, width=720 if "landscape" in row["video"] else 340))

## 8 · Download the video and the project bundle

In [ ]:
from google.colab import files   # only exists inside Colab

for row in nb.result_table(result):
    files.download(row["path"])

bundle = nb.zip_project(project)
print("bundle:", bundle, round(bundle.stat().st_size / 1e6, 1), "MB")
files.download(str(bundle))

## 9 · Resume after a disconnect

Colab recycles runtimes. The project folder is the state: script, audio, and finished scene clips all live on disk.

1. Re-run cells **0 → 2** (code + install + health check).
2. Run the cell below instead of creating a new project.
3. `render()` skips every scene that is already finished and still matches its inputs.

> If you mounted Google Drive, keep projects in `/content/drive/MyDrive/ckd-projects` so they survive a runtime reset. Change `projects_dir` in cell 3 accordingly.

In [ ]:
import json, pathlib
from ckd.notebook_api import open_project

existing = sorted(str(p.parent) for p in pathlib.Path("/content/projects").glob("*/project.json"))
print("projects on disk:")
for path in existing:
    print(" ", path)

if existing:
    project = open_project(existing[-1], root="/content")
    state = json.loads((project.root / "project.json").read_text())
    print("\nresumed:", project.root)
    print("status:", state.get("status"), "| scenes done:", len(list((project.root / "scenes").rglob("*.mp4"))))
    result = nb.render(project)          # only the missing scenes render
    display(pd.DataFrame(nb.result_table(result)))

## Troubleshooting

| symptom | cause | fix |
|---|---|---|
| `repo not found` | `REPO_URL` empty and no files | set the URL, or upload the ZIP and unzip to `/content/CKD` |
| Urdu shows boxes/tofu | `assets/fonts` missing | re-clone the repo (fonts are committed) |
| "neural voice unavailable … espeak" | Piper download blocked | normal on restricted networks; output still has narration, the report names the engine used |
| render killed / session reset | free runtime limits | re-run cells 0–2 then cell 9; finished scenes are reused |
| `NotEnoughSpace` | Colab disk full | delete `/content/projects/*/scenes` you no longer need, or use `QUALITY="draft"` |
| video has no sound | mux step failed | check the last cell output; `report.json` lists `problems` per video |

**Never** put a GitHub token inside the notebook or the repo. See `docs/PRIVATE_REPO.md`.

---
### What this notebook will **not** do

- It will not call OpenAI / Anthropic / Gemini / any hosted inference API.
- It will not invent citations or statistics; packs contain no claims that need sources.
- It will not exceed 1080p, and it will not make a 3-minute video from a 45-second plan — the script reports the length it can actually fill.
- It will not use footage it has no rights to: every frame is drawn by code from primitives.